In [1]:
import pandas as pd
import cv2

frames1 = pd.read_csv("kaggle_dataset/labels_train.csv")
frames2 = pd.read_csv("kaggle_dataset/labels_val.csv")
frames3 = pd.read_csv("kaggle_dataset/labels_trainval.csv")

frames = pd.concat([frames1, frames2, frames3])

frames = frames[frames["class_id"] == 1]

frames = (
    frames.groupby("frame")[["xmin", "xmax", "ymin", "ymax"]]
    .apply(lambda x: x.values.tolist())
    .reset_index(name="boxes")
)



print(frames.shape)



(21883, 2)


In [2]:

frames = frames.sample(14000, random_state=42)



print(frames.shape)


(14000, 2)


In [3]:
def recalculate_bounding_box(row):
    boxes = row["boxes"]
    scaled_boxes = []
    image = cv2.imread("kaggle_dataset/images/" + row['frame'])
    h, w, _ = image.shape

    for i, box in enumerate(boxes):
        xmin = box[0]
        xmax = box[1]
        ymin = box[2]
        ymax = box[3]

        
        new_xmin = int(xmin * (128 / w))
        new_ymin = int(ymin * (128 / h))
        new_xmax = int(xmax * (128 / w))
        new_ymax = int(ymax * (128 / h))

        scaled_boxes.append([new_xmin, new_xmax, new_ymin, new_ymax])

    return scaled_boxes

# test aşamasın iou için en büyük nesneyi temel alalım. Yani ilk sırada olacak olan
def calculate_box_area(box):
    xmin, xmax, ymin, ymax = box

    x = xmax-xmin
    y = ymax-ymin

    return x*y

for i, row in frames.iterrows():
    boxes = recalculate_bounding_box(row)
    
    unique = []
    for item in boxes:
        if item not in unique:
            unique.append(item)


    boxes.sort(key=calculate_box_area)
    frames.at[i,"boxes"] = unique

print(frames["boxes"])



7097                   [[5, 32, 55, 87], [53, 57, 63, 69]]
7348     [[1, 16, 63, 75], [30, 34, 63, 68], [30, 34, 6...
3115     [[34, 48, 56, 72], [54, 64, 56, 70], [63, 72, ...
7667                  [[35, 43, 62, 70], [45, 56, 59, 75]]
15702    [[64, 66, 58, 61], [69, 74, 59, 66], [81, 85, ...
                               ...                        
1550     [[3, 36, 54, 81], [70, 73, 61, 66], [72, 76, 6...
1009     [[11, 19, 60, 68], [38, 42, 61, 66], [58, 66, ...
17122    [[0, 9, 70, 98], [20, 24, 62, 65], [24, 28, 61...
10939                [[78, 90, 57, 75], [81, 100, 61, 81]]
16659    [[5, 40, 57, 92], [34, 38, 61, 69], [38, 41, 6...
Name: boxes, Length: 14000, dtype: object


In [4]:
import numpy as np

# 70 train 15 val 15 test

frames_train = frames.sample(frac=0.7, random_state=46)
frames_val = frames[~frames.index.isin(frames_train.index)].sample(frac=0.5, random_state=46)
frames_test = frames[~frames.index.isin(pd.concat([frames_train, frames_val]).index)]

np.save("train_frames.npy", frames_train.to_numpy(dtype=object))
np.save("val_frames.npy", frames_val.to_numpy(dtype=object))
np.save("test_frames.npy", frames_test.to_numpy(dtype=object))

print(len(frames_train))
print(len(frames_val))
print(len(frames_test))



9800
2100
2100


In [5]:
import os

os.makedirs("images", exist_ok= True)

for i, row in frames.iterrows():
    image = cv2.imread("kaggle_dataset/images/" + row['frame'], cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128,128))
    cv2.imwrite("images/" + row['frame'], image)

